# NB_DOC_02 · Redactar las descripciones con Claude (opcional)

**Paso 2 del framework, automatizado dentro de Fabric.** Si preferís hacerlo conversando con Claude (Claude Code o claude.ai con la skill `documentar-modelo-semantico`), salteá este notebook.

Lee `modelo.json` y `descripciones.csv` del paso 1, le pasa a Claude **una tabla por llamada** (metadatos, perfil, ejemplos no sensibles, relaciones y medidas de esa tabla) y completa `descripcion_nueva` y `sensibilidad`. **No toca el modelo** y deja todo en `revisado = no`: una persona revisa antes del paso 3.

⚠️ **Datos que salen de Fabric**: se envían a la API de Anthropic los nombres de objetos, expresiones DAX, estadísticas y hasta 3 valores de ejemplo por columna no sensible. Si eso no está permitido para este modelo, poné `ENVIAR_EJEMPLOS = False` o usá el camino manual.

La clave de API se lee de **Azure Key Vault**; nunca la pegues en el notebook.

In [ ]:
MODELO = "<nombre-del-modelo-semantico>"
CARPETA_SALIDA = "/lakehouse/default/Files/doc_modelos"
KEY_VAULT_URL = "https://<tu-key-vault>.vault.azure.net/"
SECRETO_API_KEY = "<nombre-del-secreto-con-la-api-key>"
CLAUDE_MODEL = "claude-opus-5-5"
ENVIAR_EJEMPLOS = True
CONTEXTO_NEGOCIO = """
[Describí en 2 o 3 líneas el negocio y para qué se usa el modelo. Mejora mucho las descripciones.]
"""

In [ ]:
%pip install -q -U anthropic

In [ ]:
import json, os, re
import pandas as pd
import anthropic
import notebookutils

carpeta = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", MODELO))
with open(os.path.join(carpeta, "modelo.json"), encoding="utf-8") as f:
    modelo = json.load(f)
ruta_csv = os.path.join(carpeta, "descripciones.csv")
desc = pd.read_csv(ruta_csv, dtype=str, keep_default_na=False)

client = anthropic.Anthropic(api_key=notebookutils.credentials.getSecret(KEY_VAULT_URL, SECRETO_API_KEY))

In [ ]:
# Mismas reglas que la skill documentar-modelo-semantico
SISTEMA = """Sos un arquitecto de datos que documenta modelos semánticos de Power BI / Microsoft Fabric.
Escribís en español rioplatense, claro y breve (una o dos oraciones por objeto), pensando en un usuario de negocio que arma reportes.

Reglas:
- Describí qué representa el objeto en el negocio y, si ayuda, su granularidad, unidad o regla de cálculo.
- Para medidas, explicá qué calcula en palabras a partir de la expresión DAX; no repitas la fórmula.
- Para columnas clave o de relación, decí con qué tabla se relacionan.
- Si los metadatos no alcanzan para saber qué significa algo, escribí exactamente "[dato a confirmar]" en vez de inventar.
- Sensibilidad: "Sensible" si contiene datos personales o confidenciales (identificadores de personas, contacto, salud, salarios, datos financieros personales), "Interna" si es de uso interno del negocio, "Pública" solo si es claramente pública.
- Si la columna viene marcada sensible_sugerido = true, la sensibilidad es "Sensible" salvo que sea evidente que no lo es.
"""

ESQUEMA = {
    "type": "object",
    "properties": {
        "descripcion_tabla": {"type": "string"},
        "columnas": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "nombre": {"type": "string"},
                "descripcion": {"type": "string"},
                "sensibilidad": {"type": "string", "enum": ["Pública", "Interna", "Sensible"]},
            },
            "required": ["nombre", "descripcion", "sensibilidad"],
            "additionalProperties": False,
        }},
        "medidas": {"type": "array", "items": {
            "type": "object",
            "properties": {"nombre": {"type": "string"}, "descripcion": {"type": "string"}},
            "required": ["nombre", "descripcion"],
            "additionalProperties": False,
        }},
    },
    "required": ["descripcion_tabla", "columnas", "medidas"],
    "additionalProperties": False,
}


def contexto_tabla(t):
    cols = []
    for c in t["columnas"]:
        c = dict(c)
        if not ENVIAR_EJEMPLOS or c.get("sensible_sugerido"):
            c.pop("ejemplos", None); c.pop("min", None); c.pop("max", None)
        cols.append(c)
    return {
        "tabla": {k: v for k, v in t.items() if k != "columnas"},
        "columnas": cols,
        "medidas": [m for m in modelo["medidas"] if m["tabla"] == t["nombre"]],
        "relaciones": [r for r in modelo["relaciones"]
                       if t["nombre"] in (r["desde_tabla"], r["hacia_tabla"])],
        "otras_tablas_del_modelo": [x["nombre"] for x in modelo["tablas"] if x["nombre"] != t["nombre"]],
    }


def redactar(t):
    prompt = (f"Contexto del negocio:\n{CONTEXTO_NEGOCIO.strip()}\n\n"
              f"Modelo: {modelo['modelo']}\n\n"
              "Documentá esta tabla, todas sus columnas y todas sus medidas:\n"
              + json.dumps(contexto_tabla(t), ensure_ascii=False, indent=1, default=str))
    # fallbacks="default": si el modelo rechaza la consulta por política, la API la reintenta con otro modelo
    with client.beta.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=32000,
        system=SISTEMA,
        messages=[{"role": "user", "content": prompt}],
        output_config={"effort": "medium", "format": {"type": "json_schema", "schema": ESQUEMA}},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    ) as stream:
        resp = stream.get_final_message()
    if resp.stop_reason == "refusal":
        raise RuntimeError(f"Claude no respondió para {t['nombre']}: {resp.stop_details}")
    if resp.stop_reason == "max_tokens":
        raise RuntimeError(f"Respuesta cortada para {t['nombre']}: subí max_tokens o partí la tabla")
    texto = next(b.text for b in resp.content if b.type == "text")
    return json.loads(texto)

In [ ]:
def completar(tipo, tabla, objeto, descripcion, sensibilidad=None):
    """Solo completa filas sin descripcion_nueva: nunca pisa lo que ya revisó una persona."""
    m = (desc["tipo_objeto"] == tipo) & (desc["tabla"] == tabla) & (desc["objeto"] == objeto) \
        & (desc["descripcion_nueva"] == "")
    desc.loc[m, "descripcion_nueva"] = descripcion
    if sensibilidad:
        # Solo sube la sensibilidad: una columna ya marcada "Sensible" no se baja automáticamente
        sube = m & ((desc["sensibilidad"] == "") | (sensibilidad == "Sensible"))
        desc.loc[sube, "sensibilidad"] = sensibilidad


for t in modelo["tablas"]:
    pendientes = desc[(desc["tabla"] == t["nombre"]) & (desc["descripcion_nueva"] == "")]
    if pendientes.empty:
        continue
    print("Redactando", t["nombre"], "...")
    r = redactar(t)
    completar("tabla", t["nombre"], t["nombre"], r["descripcion_tabla"])
    for c in r["columnas"]:
        completar("columna", t["nombre"], c["nombre"], c["descripcion"], c["sensibilidad"])
    for m in r["medidas"]:
        completar("medida", t["nombre"], m["nombre"], m["descripcion"])
    desc.to_csv(ruta_csv, index=False, encoding="utf-8")   # guarda después de cada tabla

faltan = (desc["descripcion_nueva"] == "").sum()
a_confirmar = desc["descripcion_nueva"].str.contains(r"\[dato a confirmar\]").sum()
print(f"Listo. Sin descripción: {faltan}. Marcadas [dato a confirmar]: {a_confirmar}.")
print("Próximo paso: revisá descripciones.csv, corregí lo necesario y poné revisado = si en lo aprobado.")